# 02 - Market-Wide Multi-Asset Portfolio Scan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huuan26/polars-anta/blob/master/examples/02_multi_asset_portfolio_scan.ipynb)

In this notebook, we demonstrate the flagship performance capability of **polars-anta**:
Computing complex indicators across an entire investment universe (500 tickers $	imes$ 1,000 bars = 500,000 rows) in **parallel across all CPU threads** using `.over("ticker")`.


In [ ]:
# Install polars-anta (uncomment if running in Colab)
!pip install -q polars polars-anta

In [ ]:
import time

import numpy as np
import polars as pl

import polars_anta

print(f"Polars Version: {pl.__version__}")
print(f"polars-anta Version: {polars_anta.__version__}")

## 1. Simulate 500 Tickers $	imes$ 1,000 Bars Universe (500,000 Rows)


In [ ]:
n_tickers = 500
n_bars = 1000
total_rows = n_tickers * n_bars

print(f"Generating universe data: {n_tickers} tickers x {n_bars} bars = {total_rows:,} rows...")
np.random.seed(42)

tickers = [f"SYM_{i:03d}" for i in range(n_tickers)]
ticker_col = np.repeat(tickers, n_bars)

close_series = []
for _ in range(n_tickers):
    ret = np.random.normal(0.0003, 0.02, n_bars)
    p = 100.0 * np.exp(np.cumsum(ret))
    close_series.append(p)

close_arr = np.concatenate(close_series)
high_arr = close_arr * (1.0 + np.abs(np.random.normal(0, 0.008, total_rows)))
low_arr = close_arr * (1.0 - np.abs(np.random.normal(0, 0.008, total_rows)))
vol_arr = np.random.randint(1000, 100000, total_rows).astype(float)

universe_df = pl.DataFrame(
    {
        "ticker": ticker_col,
        "high": high_arr,
        "low": low_arr,
        "close": close_arr,
        "volume": vol_arr,
    }
)

print(f"Universe DataFrame: {universe_df.shape[0]:,} rows x {universe_df.shape[1]} cols")
print(universe_df.head(6))

## 2. Ultra-Fast Parallel Feature Extraction with `.over('ticker')`

We compute 7 indicators simultaneously across all 500 tickers in parallel:


In [ ]:
t0 = time.perf_counter()

# Polars distributes computation across all CPU threads via Rust Rayon threadpool
scan_df = universe_df.with_columns(
    pl.col("close").ta.sma(20).over("ticker").alias("sma_20"),
    pl.col("close").ta.ema(50).over("ticker").alias("ema_50"),
    pl.col("close").ta.rsi(14).over("ticker").alias("rsi_14"),
    pl.col("close").ta.bollinger_bands(20, 2.0).over("ticker").alias("bb"),
    pl.col("close").ta.obv("volume").over("ticker").alias("obv"),
    pl.col("close").ta.zscore(20).over("ticker").alias("zscore_20"),
    (pl.col("volume") / pl.col("volume").rolling_mean(20).over("ticker")).alias("rvol"),
).unnest("bb")

elapsed = time.perf_counter() - t0
print(f"Done in {elapsed * 1000:.2f} ms ({elapsed:.4f} seconds) across {total_rows:,} rows!")
print(f"Throughput: {total_rows / elapsed:,.0f} rows/second")

## 3. High-Conviction Breakout Scanner

Screen for stocks meeting strict technical criteria on the latest bar:
1. **Price Momentum**: Breaking above Upper Bollinger Band (`close > upper`)
2. **Trend Confirmation**: Fast SMA > Slow EMA (`sma_20 > ema_50`)
3. **Volume Spike**: Relative Volume $\ge 1.5$x 20-day average (`rvol >= 1.5`)
4. **Not Overbought**: RSI $< 75$


In [ ]:
# Filter the latest bar for each ticker
latest_bars = scan_df.group_by("ticker").last()

# Apply quantitative filter
breakout_candidates = (
    latest_bars.filter(
        (pl.col("close") > pl.col("upper"))
        & (pl.col("sma_20") > pl.col("ema_50"))
        & (pl.col("rvol") >= 1.5)
        & (pl.col("rsi_14") < 75.0)
    )
    .select(["ticker", "close", "upper", "rsi_14", "rvol", "zscore_20"])
    .sort("rvol", descending=True)
)

print(f"Found {len(breakout_candidates)} Breakout Candidates:")
print(breakout_candidates)